<a href="https://colab.research.google.com/github/Rakesh114166/Rakesh-dock/blob/main/debugging_the_third_attempt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title CELL 1: Robust System Environment Setup { display-mode: "form" }
#@markdown Run this cell to completely purge cached package conflicts and deploy the clean computational informatics stack.

import sys
import subprocess
import os

# --- Step 1: PURGE CACHE AND LEGACY PACKAGES ---
# Removes old configurations and broken source installations that prevent smooth deployment.
print("[-] Clearing cached package configurations and broken artifacts...")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "openbabel", "openbabel-wheel", "plip", "rdkit", "py3Dmol", "pandas"], check=False)
subprocess.run(["sudo", "apt-get", "purge", "-y", "openbabel", "libopenbabel-dev"], check=False)

# --- Step 2: UPDATE SYSTEM REGISTRIES & BUILD TOOLS ---
# Updates the base OS registry and, CRITICALLY, modernizes Python's core build ecosystem.
print("[-] Updating system registries and modernizing core Python build tools...")
subprocess.run(["sudo", "apt-get", "update", "-qq"], check=True)
# libopenbabel-dev and swig are often required by PLIP to compile specific extensions seamlessly.
subprocess.run(["sudo", "apt-get", "install", "-y", "-qq", "openbabel", "libopenbabel-dev", "swig"], check=True)
# Explicitly upgrading pip, setuptools, and wheel prevents standard Status 1 install errors on any machine.
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], check=True)

# --- Step 3: STATIC BINARY DEPLOYMENT (AutoDock Vina) ---
# Deploys the uncompressed static Vina binary directly into the execution PATH.
print("[-] Deploying static binary for AutoDock Vina v1.2.5...")
vina_url = "https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64"
vina_path = "/usr/local/bin/vina"
subprocess.run(["wget", "-q", vina_url, "-O", vina_path], check=True)
subprocess.run(["chmod", "+x", vina_path], check=True)

# --- Step 4: CORE INFORMATICS STACK & PLIP DEPLOYMENT ---
# Installs packages without silencing output so errors can be instantly diagnosed.
print("[-] Deploying foundational chemical informatics stack (RDKit, py3Dmol, pandas)...")
# Removed quiet flag "-q" for robustness.
subprocess.run([sys.executable, "-m", "pip", "install", "rdkit", "py3Dmol", "pandas"], check=True)

print("[-] Deploying stable pre-compiled OpenBabel wheels...")
# Installs openbabel-wheel first to satisfy PLIP's dependency without broken source compilation.
subprocess.run([sys.executable, "-m", "pip", "install", "openbabel-wheel"], check=True)

print("[-] Deploying interaction profiler engine (PLIP) via standard dependency resolution...")
# Installed normally, pip will recognize openbabel-wheel satisfies the dependency.
subprocess.run([sys.executable, "-m", "pip", "install", "plip"], check=True)

print("\n[+] SYSTEM CONFIGURATION DEPLOYED SUCCESSFULLY")
print(f"    ↳ Vina Version: {subprocess.getoutput('vina --version').splitlines()[0]}")
print(f"    ↳ Babel CLI: {subprocess.getoutput('obabel -V').strip()}")

[-] Clearing cached package configurations and broken artifacts...
[-] Updating system registries and modernizing core Python build tools...
[-] Deploying static binary for AutoDock Vina v1.2.5...
[-] Deploying foundational chemical informatics stack (RDKit, py3Dmol, pandas)...
[-] Deploying stable pre-compiled OpenBabel wheels...
[-] Deploying interaction profiler engine (PLIP) via standard dependency resolution...

[+] SYSTEM CONFIGURATION DEPLOYED SUCCESSFULLY
    ↳ Vina Version: AutoDock Vina v1.2.5
    ↳ Babel CLI: Open Babel 3.2.0 -- May 26 2026 -- 19:22:48


In [2]:
#@title CELL 2: Core Module Initialization { display-mode: "form" }
#@markdown Initializes standard, specialized, and structural module dependencies into the namespace.

import os
import sys
import subprocess

print("[-] Verifying core ecosystem stability and binding paths...")

try:
    import pandas as pd
    import py3Dmol
    import rdkit
    from rdkit import Chem
    from rdkit.Chem import AllChem
    import plip
    from plip.structure.preparation import PDBComplex

    print("\n[+] CORE ENVIRONMENT INITIALIZED SUCCESSFULLY")
    print(f"    ↳ Interpreter Platform   : Python {sys.version.split()[0]}")
    print(f"    ↳ Pandas Data Matrix API : v{pd.__version__}")
    print(f"    ↳ RDKit Core Engine      : v{rdkit.__version__}")
    print(f"    ↳ PLIP Profiler Binding  : Active and stable")

except Exception as e:
    print(f"\n[!] Initialization Failure: {e}")
    print("    ↳ Action Required: Go to the top menu, click 'Runtime' -> 'Restart Session', then re-run CELL 1 and CELL 2.")

[-] Verifying core ecosystem stability and binding paths...

[+] CORE ENVIRONMENT INITIALIZED SUCCESSFULLY
    ↳ Interpreter Platform   : Python 3.12.13
    ↳ Pandas Data Matrix API : v3.0.3
    ↳ RDKit Core Engine      : v2026.03.2
    ↳ PLIP Profiler Binding  : Active and stable


In [3]:
#@title CELL 3: Project Directory Layout Generator { display-mode: "form" }
#@markdown Enter a `< Job Name >` without spaces to establish your workspace directories.

Job_name = "1MOX_VS" #@param {type:"string"}

import os

# Strict input verification matching the original layout rules
assert not Job_name == "", "Do not leave this blank."
assert not any(c == "/" or c == "." or c == " " for c in Job_name), "Disallowed characters detected in your Job Name."

print(f"[-] Initializing project workspace root for simulation tracking: '{Job_name}'")

# Map out absolute directory paths matching your workflow chart layout
BASE_DIR = os.getcwd()
WRK_DIR  = os.path.join(BASE_DIR, Job_name)
PRT_FLD  = os.path.join(WRK_DIR, "PROTEIN")
LIG_FLD  = os.path.join(WRK_DIR, "LIGAND")
EXP_FLD  = os.path.join(WRK_DIR, "EXPERIMENTAL")
DCK_FLD  = os.path.join(WRK_DIR, "DOCKING")
INT_FLD  = os.path.join(WRK_DIR, "INTERACTION")

folders = [WRK_DIR, PRT_FLD, LIG_FLD, EXP_FLD, DCK_FLD, INT_FLD]

for f in folders:
    if os.path.exists(f):
        print(f"    [!] Notice: Node element folder already exists -> {os.path.basename(f)}")
    else:
        os.makedirs(f, exist_ok=True)
        print(f"    [->] Successfully Generated Directory Folder Node: {f}")

# Inject path mappings into environmental variables for clean cross-cell data handshake
os.environ['WRK_DIR'] = WRK_DIR
os.environ['PRT_FLD'] = PRT_FLD
os.environ['LIG_FLD'] = LIG_FLD
os.environ['EXP_FLD'] = EXP_FLD
os.environ['DCK_FLD'] = DCK_FLD
os.environ['INT_FLD'] = INT_FLD

print("\n[+] WORKSPACE PATHWAYS ASSIGNED SUCCESSFULLY")
print(f"    ↳ Active Project Root Directory: {os.environ['WRK_DIR']}")

[-] Initializing project workspace root for simulation tracking: '1MOX_VS'
    [->] Successfully Generated Directory Folder Node: /content/1MOX_VS
    [->] Successfully Generated Directory Folder Node: /content/1MOX_VS/PROTEIN
    [->] Successfully Generated Directory Folder Node: /content/1MOX_VS/LIGAND
    [->] Successfully Generated Directory Folder Node: /content/1MOX_VS/EXPERIMENTAL
    [->] Successfully Generated Directory Folder Node: /content/1MOX_VS/DOCKING
    [->] Successfully Generated Directory Folder Node: /content/1MOX_VS/INTERACTION

[+] WORKSPACE PATHWAYS ASSIGNED SUCCESSFULLY
    ↳ Active Project Root Directory: /content/1MOX_VS


In [47]:
#@title CELL 4: Receptor Chain Isolation & Parameterization { display-mode: "form" }
#@markdown Download target PDB structures, isolate desired protein chains, strip water matrices, and parameterize to PDBQT.

PDB_ID = "1M17" #@param {type:"string"}
CHAIN_ID = "A" #@param {type:"string"}

import os
import urllib.request
import subprocess
import py3Dmol

# Fetch target directories from the environment variables set in Cell 3
PRT_FLD = os.environ.get('PRT_FLD')
DCK_FLD = os.environ.get('DCK_FLD')

if not PRT_FLD or not DCK_FLD:
    raise ValueError("Workspace directories not detected. Please run Cell 3 first to generate paths.")

# Dynamically construct filenames to match your workflow schema exactly
base_name = f"{PDB_ID.upper()}_prot_{CHAIN_ID.upper()}"
raw_pdb_path = os.path.join(PRT_FLD, f"{PDB_ID.lower()}.pdb")
clean_pdb_path = os.path.join(PRT_FLD, f"{base_name}.pdb")
docking_pdb_copy = os.path.join(DCK_FLD, f"{base_name}.pdb")
receptor_pdbqt = os.path.join(DCK_FLD, f"{base_name}.pdbqt")

print(f"[-] Fetching crystal structure '{PDB_ID.upper()}' securely from RCSB PDB Repository...")
url = f"https://files.rcsb.org/download/{PDB_ID.upper()}.pdb"
try:
    urllib.request.urlretrieve(url, raw_pdb_path)
    print(f"    ↳ PDB downloaded: {PDB_ID.lower()}.pdb ==> PROTEIN folder")
except Exception as e:
    raise RuntimeError(f"Network error: Unable to retrieve PDB file {PDB_ID}. Details: {e}")

print(f"[-] Extracting protein: Isolating Chain '{CHAIN_ID}' and clearing water matrices...")
atom_count = 0

with open(raw_pdb_path, 'r') as infile, open(clean_pdb_path, 'w') as outfile:
    for line in infile:
        # Capture standard amino acid backbone rows (ATOM)
        if line.startswith("ATOM  "):
            line_chain = line[21].strip()
            # If a specific chain is required, match it; otherwise extract all chains
            if line_chain == CHAIN_ID or not CHAIN_ID:
                outfile.write(line)
                atom_count += 1
        # Maintain core structural break coordinates
        elif line.startswith("TER   ") or line.startswith("ENDMDL"):
            if atom_count > 0:  # Only write breaks if we have already isolated atoms for this frame
                outfile.write(line)

if atom_count == 0:
    raise ValueError(f"Target isolation failed: No ATOM records found matching Chain '{CHAIN_ID}'.")
print(f"    ↳ Protein extracted: {PDB_ID.lower()}.pdb --> {base_name}.pdb")

# Sync clean PDB file over to the DOCKING folder to match original tutorial architecture
with open(clean_pdb_path, 'r') as src, open(docking_pdb_copy, 'w') as dst:
    dst.write(src.read())

print("[-] Running structural parameterization and Gasteiger charge assignment via OpenBabel...")
# -xr builds a rigid receptor by preserving atom records and blocking bond rotations
cmd = ["obabel", clean_pdb_path, "-O", receptor_pdbqt, "-xr", "--partialcharge", "gasteiger"]
subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print(f"    ↳ Protein parameterized: {base_name}.pdb --> {base_name}.pdbqt")
print(f"    ↳ {base_name}.pdb ==> DOCKING folder")
print(f"    ↳ {base_name}.pdbqt ==> DOCKING folder")

# Render the isolated receptor backbone interactively
print("\n[-] Displaying 3D protein macromolecular architecture...")
view = py3Dmol.view(width=650, height=450)
with open(clean_pdb_path, 'r') as f:
    view.addModel(f.read(), 'pdb')
view.setStyle({'cartoon': {'color': 'spectrum'}})
view.zoomTo()
view.show()

[-] Fetching crystal structure '1M17' securely from RCSB PDB Repository...
    ↳ PDB downloaded: 1m17.pdb ==> PROTEIN folder
[-] Extracting protein: Isolating Chain 'A' and clearing water matrices...
    ↳ Protein extracted: 1m17.pdb --> 1M17_prot_A.pdb
[-] Running structural parameterization and Gasteiger charge assignment via OpenBabel...
    ↳ Protein parameterized: 1M17_prot_A.pdb --> 1M17_prot_A.pdbqt
    ↳ 1M17_prot_A.pdb ==> DOCKING folder
    ↳ 1M17_prot_A.pdbqt ==> DOCKING folder

[-] Displaying 3D protein macromolecular architecture...


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [49]:
#@title CELL 5: Experimental Native Control Extraction { display-mode: "form" }
#@markdown Enter the official 3-letter residue code for the co-crystallized control drug present inside the target crystal structure (e.g., NAG, PY1, STI).

Crystallized_Ligand_ID = "AQ4" #@param {type:"string"}

import os
import subprocess

# Retrieve absolute directory pathways from environmental memory set in Cell 3
PRT_FLD = os.environ.get('PRT_FLD')
EXP_FLD = os.environ.get('EXP_FLD')
DCK_FLD = os.environ.get('DCK_FLD')

if not all([PRT_FLD, EXP_FLD, DCK_FLD]):
    raise ValueError("Workspace pathways missing. Please re-run Cell 3 first.")

# Standardize input code token string to upper-case matching standard PDB formatting conventions
lig_code = Crystallized_Ligand_ID.strip().upper()

# Clear out any stale historical control structures inside the docking track workspace
for folder in [EXP_FLD, DCK_FLD]:
    for f in os.listdir(folder):
        if f.endswith(('.pdb', '.pdbqt')) and not "_prot_" in f:
            os.remove(os.path.join(folder, f))

# Detect source macro-structural coordinate target file generated by Cell 4
raw_pdb_candidates = [f for f in os.listdir(PRT_FLD) if f.endswith('.pdb') and not '_prot_' in f]

if not raw_pdb_candidates:
    print("[!] Error: No baseline reference PDB files discovered inside the PROTEIN workspace folder.")
    print("    ↳ Action required: Ensure CELL 4 executed successfully before running control extractions.")
else:
    source_pdb_path = os.path.join(PRT_FLD, raw_pdb_candidates[0])

    if not lig_code or len(lig_code) != 3:
        print("[+] Notice: Bypassing co-crystallized control drug isolation routine.")
        print("    ↳ Running pipeline in Apoprotein mode. Downstream cells will utilize manual slider coordinate boundaries.")
    else:
        print(f"[-] Scanning crystal matrices for reference control ligand token: [{lig_code}]")

        extracted_lines = []
        with open(source_pdb_path, 'r') as pdb_file:
            for line in pdb_file:
                # Intercept coordinate maps matching HETATM or ATOM configurations for the target identifier token
                if (line.startswith("HETATM") or line.startswith("ATOM")) and line[17:20].strip() == lig_code:
                    extracted_lines.append(line)

        if not extracted_lines:
            print(f"[!] Warning: Coordinate token '{lig_code}' was not found inside the active PDB structure matrix.")
            print("    ↳ Downstream grid calculations will default automatically back to manual slider configurations.")
        else:
            print(f"[+] Discovered {len(extracted_lines)} atom coordinate vectors for target ligand [{lig_code}].")

            # Destination path maps for programmatic parsing tracking
            raw_lig_pdb = os.path.join(EXP_FLD, f"{lig_code}.pdb")
            dck_lig_pdb = os.path.join(DCK_FLD, f"{lig_code}.pdb")
            dck_lig_pdbqt = os.path.join(DCK_FLD, f"{lig_code}.pdbqt")

            # Write out pure un-parameterized reference coordinate file
            with open(raw_lig_pdb, 'w') as out_f:
                out_f.writelines(extracted_lines)
                out_f.write("END\n")

            # Mirror clean coordinate blueprint into active docking track space
            with open(dck_lig_pdb, 'w') as out_f:
                out_f.writelines(extracted_lines)
                out_f.write("END\n")

            print(f"[-] Parameterizing charge topologies and protonating control drug coordinate lines at pH 7.4...")

            # Utilize OpenBabel to correctly add hydrogen geometries and calculate standard internal partial charge profiles
            cmd_babel = [
                "obabel", raw_lig_pdb,
                "-O", dck_lig_pdbqt,
                "-p", "7.4",
                "--partialcharge", "gasteiger"
            ]

            convert_job = subprocess.run(cmd_babel, capture_output=True, text=True)

            if os.path.exists(dck_lig_pdbqt) and os.path.getsize(dck_lig_pdbqt) > 0:
                print(f"\n[+] NATIVE REFERENCE BASELINE ARTIFACT GENERATED:")
                print(f"    ↳ Extracted Crystal Model   -> {raw_lig_pdb}")
                print(f"    ↳ Parameterized Vina Model -> {dck_lig_pdbqt}")
                print(f"    ↳ Configuration Status      -> Ready for downstream grid alignment visualization.")
            else:
                print(f"[!] Critical Parameterization Failure. OpenBabel logging output:\n{convert_job.stderr}")

[-] Scanning crystal matrices for reference control ligand token: [AQ4]
[+] Discovered 29 atom coordinate vectors for target ligand [AQ4].
[-] Parameterizing charge topologies and protonating control drug coordinate lines at pH 7.4...

[+] NATIVE REFERENCE BASELINE ARTIFACT GENERATED:
    ↳ Extracted Crystal Model   -> /content/1MOX_VS/EXPERIMENTAL/AQ4.pdb
    ↳ Parameterized Vina Model -> /content/1MOX_VS/DOCKING/AQ4.pdbqt
    ↳ Configuration Status      -> Ready for downstream grid alignment visualization.


In [50]:
#@title CELL 6: CSV High-Throughput Ligand Preparation { display-mode: "form" }
#@markdown Batch-process an uploaded spreadsheet of compounds from SMILES to optimized 3D PDBQT structures.

Compound_library = "MONAMI.csv" #@param {type:"string"}
Select_force_field = "UFF" #@param ["GAFF", "Ghemical", "MMFF94", "MMFF94s", "UFF"]
Max_minimization_steps = 10000 #@param {type:"slider", min:1000, max:20000, step:1000}
View_ligand = "SKQR1" #@param {type:"string"}

import os
import shutil
import subprocess
import pandas as pd
import py3Dmol

# Retrieve absolute directory trees from the environment variables set in Cell 3
WRK_DIR = os.environ.get('WRK_DIR')
LIG_FLD = os.environ.get('LIG_FLD')
DCK_FLD = os.environ.get('DCK_FLD')

if not all([WRK_DIR, LIG_FLD, DCK_FLD]):
    raise ValueError("Workspace pathways missing. Please re-run Cell 3 first.")

CL_csv_Lfile = os.path.join(LIG_FLD, Compound_library)

# --- AUTOMATED TEMPLATE GENERATOR ---
if not os.path.exists(CL_csv_Lfile):
    print(f"[!] Alert: Spreadsheet template '{Compound_library}' not detected inside LIGAND folder.")
    print("    ↳ Generating a fresh screening library containing PY1 (control), Naringenin, and Esculin...")
    sample_payload = {
        "ID": ["PY1", "NARINGENIN", "ESCULIN"],
        "SMILES": [
            "C1=CC=C2C(=C1)C(=CC=N2)C3=CNN=C3C4=CC=CC=N4", # PY1 (True 1PY5 Control Drug)
            "O=C1CC(C2=CC=C(O)C=C2)OC3=CC(O)=CC(O)=C13",     # Naringenin
            "O=C1C=C(O[C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]2O)C3=C(O)C=CC(=C3O1)" # Esculin
        ]
    }
    pd.DataFrame(sample_payload).to_csv(CL_csv_Lfile, index=False)

# Read library using the exact string column indexing from your video layout
try:
    ligDF = pd.read_csv(CL_csv_Lfile, dtype=str)
    lig_IDs = tuple(ligDF["ID"].str.strip().str.upper())
    lig_SMILES = tuple(ligDF["SMILES"].str.strip())
except Exception as e:
    raise RuntimeError(f"Failed to read chemical library layout matrix: {e}")

print(f"[+] Loaded chemical target library spreadsheet. Processing {len(lig_IDs)} compounds...")

# Step 1: Write raw .smi structural seed rows to disk
for N, ID in enumerate(lig_IDs):
    smi_file = os.path.join(LIG_FLD, f"{ID}.smi")
    with open(smi_file, "w") as o:
        o.write(lig_SMILES[N])

print(f"    ↳ + {len(lig_IDs)} ligand.smi ==> LIGAND folder")

# Step 2: High-Performance 3D Coordinate Mapping & Charge Parameterization Loop
for N, ID in enumerate(lig_IDs):
    smi_file = os.path.join(LIG_FLD, f"{ID}.smi")
    mol2_file = os.path.join(WRK_DIR, f"{ID}.mol2")
    final_mol2_path = os.path.join(LIG_FLD, f"{ID}.mol2")

    # Establish separate subfolders for each ligand inside DOCKING
    lig_Dfld = os.path.join(DCK_FLD, ID)
    os.makedirs(lig_Dfld, exist_ok=True)
    pdbqt_file = os.path.join(lig_Dfld, f"{ID}.pdbqt")

    # Run structural minimization macro-loop via modern OpenBabel CLI
    # Generates lowest-energy 3D conformations using your chosen force field selection
    cmd_minimize = [
        "obabel", smi_file, "-O", mol2_file,
        "--gen3d", "--best", "--canonical",
        "--minimize", "--ff", Select_force_field,
        "--steps", str(Max_minimization_steps)
    ]
    subprocess.run(cmd_minimize, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # Move minimized .mol2 transition file to LIGAND folder to preserve workspace architecture
    if os.path.exists(mol2_file):
        shutil.move(mol2_file, final_mol2_path)

    # Parameterize .mol2 structure to target .pdbqt docking profile with Gasteiger charges at pH 7.4
    cmd_parameterize = [
        "obabel", final_mol2_path, "-O", pdbqt_file,
        "-ph", "7.4", "--partialcharge", "gasteiger"
    ]
    subprocess.run(cmd_parameterize, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print(f"    ↳ + {len(lig_IDs)} ligand.mol2 ==> LIGAND folder")
print(f"    ↳ + {len(lig_IDs)} ligand.pdbqt ==> DOCKING folder subdirectories")
print(f"\n[+] HIGH-THROUGHPUT LIGAND PREPARATION COMPLETE USING FORCE FIELD: {Select_force_field}")

# Step 3: Interactive 3D Conformer Viewer Panel
target_view_mol2 = os.path.join(LIG_FLD, f"{View_ligand.strip().upper()}.mol2")

if os.path.exists(target_view_mol2):
    print(f"\n[-] Displaying 3D optimized chemical conformation for: {View_ligand.upper()}")
    view = py3Dmol.view(width=650, height=450)
    with open(target_view_mol2, 'r') as f:
        view.addModel(f.read(), 'mol2')
    view.setStyle({'stick': {'colorscheme': 'greenCarbon'}})
    view.zoomTo()
    view.show()
else:
    print(f"\n[!] View Alert: Cannot display 3D structure for '{View_ligand.upper()}'. ID not found in library.")

[+] Loaded chemical target library spreadsheet. Processing 2 compounds...
    ↳ + 2 ligand.smi ==> LIGAND folder
    ↳ + 2 ligand.mol2 ==> LIGAND folder
    ↳ + 2 ligand.pdbqt ==> DOCKING folder subdirectories

[+] HIGH-THROUGHPUT LIGAND PREPARATION COMPLETE USING FORCE FIELD: UFF

[-] Displaying 3D optimized chemical conformation for: SKQR1


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [52]:
#@title CELL 7: Place Grid Box at Binding Site { display-mode: "form" }
#@markdown Establish the grid search box parameters over the target binding pocket and configure the 3D visualization studio canvas themes.

Focus_residues = "" #@param {type:"string"}
Method = "eBoxSize-modified" #@param ["LaBOX", "eBoxSize", "eBoxSize-modified", "Autodock-Grid", "manual"]

#@markdown --- 🎨 Studio Display Customizations ---
Grid_Box_Color = "purple" #@param ["purple", "green", "blue", "red", "yellow", "cyan", "orange"]
Show_Crystallized_Ligand = True #@param {type:"boolean"}
Protein_Display_Style = "cartoon" #@param ["cartoon", "line", "stick", "sphere", "None"]
Protein_Display_Color = "yellow" #@param ["grey", "spectrum", "blue", "red", "green", "yellow"]
Protein_Opacity = 0.7 #@param {type:"slider", min:0.1, max:1.0, step:0.05}

#@markdown --- ⚙️ Manual Mode Adjustments (Active only if Method is set to 'manual') ---
X = -8.8 #@param {type:"slider", min:-100, max:100, step:0.1}
Y = 66.6 #@param {type:"slider", min:-100, max:100, step:0.1}
Z = 20.5 #@param {type:"slider", min:-100, max:100, step:0.1}
Width = 20 #@param {type:"slider", min:10, max:40, step:0.5}
Height = 20 #@param {type:"slider", min:10, max:40, step:0.5}
Depth = 20 #@param {type:"slider", min:10, max:40, step:0.5}

import os
import math
import py3Dmol

# Retrieve absolute directory trees from environmental memory
DCK_FLD = os.environ.get('DCK_FLD')
if not DCK_FLD:
    raise ValueError("Workspace pathways missing. Please re-run Cell 3 first.")

# Locate files inside docking directory track workspace
lig_candidates = [f for f in os.listdir(DCK_FLD) if f.endswith('.pdb') and not '_prot_' in f]
prot_candidates = [f for f in os.listdir(DCK_FLD) if f.endswith('.pdb') and "_prot_" in f]

cx, cy, cz = X, Y, Z
sx, sy, sz = Width, Height, Depth

if Method != "manual":
    if not lig_candidates:
        print("[!] Warning: Reference control file missing from directory. Defaulting to manual coordinates.")
        Method = "manual"
    else:
        control_path = os.path.join(DCK_FLD, lig_candidates[0])
        print(f"[-] Parsing coordinates from reference control file: {os.path.basename(control_path)}")

        x_coords, y_coords, z_coords = [], [], []
        with open(control_path, 'r') as f:
            for line in f:
                if line.startswith("HETATM") or line.startswith("ATOM"):
                    try:
                        x_coords.append(float(line[30:38].strip()))
                        y_coords.append(float(line[38:46].strip()))
                        z_coords.append(float(line[46:54].strip()))
                    except ValueError:
                        continue

        if not x_coords:
            print("[!] Warning: Reference file contained no coordinates. Falling back to manual mode.")
            Method = "manual"
        else:
            min_x, max_x = min(x_coords), max(x_coords)
            min_y, max_y = min(y_coords), max(y_coords)
            min_z, max_z = min(z_coords), max(z_coords)

            mean_x = sum(x_coords) / len(x_coords)
            mean_y = sum(y_coords) / len(y_coords)
            mean_z = sum(z_coords) / len(z_coords)

            if Method == "LaBOX":
                cx = round((max_x + min_x) / 2, 3)
                cy = round((max_y + min_y) / 2, 3)
                cz = round((max_z + min_z) / 2, 3)
                sx = round((max_x - min_x) + 4.5, 1)
                sy = round((max_y - min_y) + 4.5, 1)
                sz = round((max_z - min_z) + 4.5, 1)

            elif Method in ["eBoxSize", "eBoxSize-modified"]:
                cx, cy, cz = round(mean_x, 3), round(mean_y, 3), round(mean_z, 3)
                sq_distances = [(x - mean_x)**2 + (y - mean_y)**2 + (z - mean_z)**2 for x, y, z in zip(x_coords, y_coords, z_coords)]
                rg = math.sqrt(sum(sq_distances) / len(sq_distances))
                multiplier = 2.5 if Method == "eBoxSize" else 3.0
                calculated_dimension = round((rg * multiplier) + 4.0, 1)
                sx, sy, sz = calculated_dimension, calculated_dimension, calculated_dimension

            elif Method == "Autodock-Grid":
                cx, cy, cz = round(mean_x, 3), round(mean_y, 3), round(mean_z, 3)
                sx, sy, sz = 22.5, 22.5, 22.5

print(f"[-] Grid calculation complete using logic method: [{Method.upper()}]")

config_path = os.path.join(DCK_FLD, "grid_config.txt")
with open(config_path, 'w') as cfg:
    cfg.write(f"center_x = {cx}\ncenter_y = {cy}\ncenter_z = {cz}\n\n")
    cfg.write(f"size_x = {sx}\nsize_y = {sy}\nsize_z = {sz}\n")

print("\n[+] DOCKING SEARCH SPACE BOUNDARIES REGISTERED")
print(f"    ↳ Geometric Pocket Center : ({cx}, {cy}, {cz})")
print(f"    ↳ Envelope Dimensions (Å) : {sx} x {sy} x {sz}")

# --- WEBGL 3D BOX RENDERING CANVAS ---
print(f"\n[-] Rendering 3D Verification Canvas... Box: [{Grid_Box_Color.upper()}], Protein: [{Protein_Display_Color.upper()}]")
view = py3Dmol.view(width=700, height=500)

# Layer 1: Render the target receptor protein backbone
if prot_candidates and Protein_Display_Style != "None":
    with open(os.path.join(DCK_FLD, prot_candidates[0]), 'r') as f:
        view.addModel(f.read(), 'pdb')

    # Standard hex translation definitions (using high-vibrancy tones)
    prot_color_map = {
        "grey": {"color": "#94A3B8"},
        "spectrum": {"color": "spectrum"},
        "blue": {"color": "#2563EB"},
        "red": {"color": "#DC2626"},
        "green": {"color": "#16A34A"},
        "yellow": {"color": "#F59E0B"} # Rich Amber Gold for better visibility on white backgrounds
    }
    style_config = prot_color_map.get(Protein_Display_Color, {"color": "#CBD5E1"}).copy()

    # Assign the user-selected interactive opacity setting
    style_config["opacity"] = Protein_Opacity
    view.setStyle({'model': 0}, {Protein_Display_Style: style_config})

# Layer 2: Render crystallized control drug context (NAG / PY1)
has_lig_rendered = False
if Show_Crystallized_Ligand and lig_candidates:
    with open(os.path.join(DCK_FLD, lig_candidates[0]), 'r') as f:
        view.addModel(f.read(), 'pdb')
    lig_model_idx = 1 if (prot_candidates and Protein_Display_Style != "None") else 0
    view.setStyle({'model': lig_model_idx}, {'stick': {'colorscheme': 'magentaCarbon', 'radius': 0.25}})
    has_lig_rendered = True

# Box style color translations
box_color_map = {
    "purple": "#7E22CE", "green": "#16A34A", "blue": "#0284C7",
    "red": "#DC2626", "yellow": "#EAB308", "cyan": "#06B6D4", "orange": "#EA580C"
}
selected_box_hex = box_color_map.get(Grid_Box_Color, "#7E22CE")

# Layer 3: Render the bounding wireframe box
view.addBox({
    'center': {'x': cx, 'y': cy, 'z': cz},
    'dimensions': {'w': sx, 'h': sy, 'd': sz},
    'color': selected_box_hex,
    'opacity': 0.25 # Slightly lowered box intensity so the protein stands out clearly
})

if has_lig_rendered:
    view.zoomTo({'model': lig_model_idx})
else:
    view.zoomTo()

view.show()

[-] Parsing coordinates from reference control file: AQ4.pdb
[-] Grid calculation complete using logic method: [EBOXSIZE-MODIFIED]

[+] DOCKING SEARCH SPACE BOUNDARIES REGISTERED
    ↳ Geometric Pocket Center : (22.014, 0.253, 52.794)
    ↳ Envelope Dimensions (Å) : 18.4 x 18.4 x 18.4

[-] Rendering 3D Verification Canvas... Box: [PURPLE], Protein: [YELLOW]


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [53]:
#@title CELL 8: Perform Molecular Docking Job { display-mode: "form" }
#@markdown Launch high-throughput parallel docking routines with absolute file-sanitization and native self-docking control.

Docking_Mode = "Run All from CSV" #@param ["Run All from CSV", "Run Single Specific Ligand"]
Specific_Ligand_ID = "" #@param {type:"string"}
Redock_Crystallized_Ligand = True #@param {type:"boolean"}
Exhaustiveness = 8 #@param {type:"slider", min:1, max:128, step:1}

import os
import subprocess
import pandas as pd
import shutil

# Retrieve absolute directory pathways from environmental memory
LIG_FLD = os.environ.get('LIG_FLD')
DCK_FLD = os.environ.get('DCK_FLD')

if not LIG_FLD or not DCK_FLD:
    raise ValueError("Workspace pathways missing. Please re-run Cell 3 first.")

config_path = os.path.join(DCK_FLD, "grid_config.txt")
if not os.path.exists(config_path):
    raise FileNotFoundError("Missing grid boundary file 'grid_config.txt'. Please run Cell 7 first.")

# --- ENFORCE PDBQT RECEPTOR ACCURACY ---
prot_candidates = [f for f in os.listdir(DCK_FLD) if f.endswith('.pdbqt') and "_prot_" in f]
if not prot_candidates:
    raw_prot_candidates = [f for f in os.listdir(DCK_FLD) if f.endswith('.pdb') and "_prot_" in f]
    if not raw_prot_candidates:
        raise FileNotFoundError("Missing parameterized receptor protein backbone file. Please run Cell 4 first.")
    raw_pdb_mox = os.path.join(DCK_FLD, raw_prot_candidates[0])
    receptor_path = raw_pdb_mox + "qt"
    subprocess.run(["obabel", raw_pdb_mox, "-O", receptor_path, "-xr"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
else:
    receptor_path = os.path.join(DCK_FLD, prot_candidates[0])

print(f"[+] Verified Rigid Receptor Target ==> {os.path.basename(receptor_path)}")

# --- BULLETPROOF LIGAND FLATTENING SANITIZER ---
def clean_and_flatten_pdbqt(file_path):
    """
    Strips out multi-root structural anomalies while preserving the
    original atom coordinates exactly without column shifting.
    """
    if not os.path.exists(file_path):
        return

    with open(file_path, 'r', errors='ignore') as f:
        lines = f.readlines()

    clean_lines = ["ROOT\n"]
    for line in lines:
        if line.startswith(("ATOM", "HETATM")):
            # Clear out mathematical NaN artifacts if generated by OpenBabel
            if "nan" in line.lower():
                line = line.replace("nan", "0.00").replace("NAN", "0.00")
            clean_lines.append(line)

    clean_lines.append("ENDROOT\n")
    clean_lines.append("TORSDOF 0\n")

    with open(file_path, 'w') as f:
        f.writelines(clean_lines)

# --- BUILD RUNTIME TASK QUEUE ---
docking_queue = []

if Docking_Mode == "Run Single Specific Ligand":
    target_id = Specific_Ligand_ID.strip().upper()
    if target_id:
        docking_queue.append((target_id, os.path.join(DCK_FLD, target_id, f"{target_id}.pdbqt"), os.path.join(DCK_FLD, target_id)))
else:
    csv_candidates = [f for f in os.listdir(LIG_FLD) if f.endswith('.csv')]
    if not csv_candidates:
        raise FileNotFoundError("No library catalog registry spreadsheet (.csv) discovered inside the LIGAND folder.")
    active_csv_name = "compound_1.csv" if "compound_1.csv" in csv_candidates else csv_candidates[0]
    CL_csv_Lfile = os.path.join(LIG_FLD, active_csv_name)
    print(f"[-] Dynamic File Match: Ingesting library catalog from registry: [{active_csv_name}]")
    ligDF = pd.read_csv(CL_csv_Lfile, dtype=str)
    for ID in ligDF["ID"].str.strip().str.upper():
        docking_queue.append((ID, os.path.join(DCK_FLD, ID, f"{ID}.pdbqt"), os.path.join(DCK_FLD, ID)))

# Inject crystallized control drug targets cleanly if requested
if Redock_Crystallized_Ligand:
    control_candidates = [f for f in os.listdir(DCK_FLD) if f.endswith('.pdbqt') and not "_prot_" in f and not "_output" in f]
    if control_candidates:
        c_name = control_candidates[0].split('.')[0].upper()
        c_path = os.path.join(DCK_FLD, control_candidates[0])
        c_out_dir = os.path.join(DCK_FLD, c_name)
        os.makedirs(c_out_dir, exist_ok=True)
        shutil_target = os.path.join(c_out_dir, f"{c_name}.pdbqt")
        shutil.copy2(c_path, shutil_target)
        if (c_name, shutil_target, c_out_dir) not in docking_queue:
            docking_queue.insert(0, (c_name, shutil_target, c_out_dir))
            print(f"[+] Validation Injection: Native control ligand [{c_name}] queued for re-docking.")

print(f"[+] Vina Engine Initialized. Commencing molecular simulation tasks across {len(docking_queue)} compounds...\n")

# --- EXECUTION LOOP ---
for ID, ligand_path, target_dir in docking_queue:
    log_txt_file = os.path.join(target_dir, f"{ID}_log.txt")
    out_pdbqt_file = os.path.join(target_dir, f"{ID}_output.pdbqt")
    out_sdf_file = os.path.join(target_dir, f"{ID}_output.sdf")
    split_pdb_prefix = os.path.join(target_dir, f"{ID}_")

    if not os.path.exists(ligand_path):
        print(f"[!] Skipping Task [{ID}]: Targeted structural file missing.")
        continue

    # Standardize structural coordinate layout perfectly before running Vina
    clean_and_flatten_pdbqt(ligand_path)

    print("="*70)
    print(f" RUNNING SIMULATION FOR TARGET LIGAND COMPLEX: {ID}")
    print("="*70)

    cmd_vina = [
        "vina",
        "--receptor", receptor_path,
        "--ligand", ligand_path,
        "--config", config_path,
        "--exhaustiveness", str(Exhaustiveness),
        "--out", out_pdbqt_file
    ]

    process = subprocess.run(cmd_vina, capture_output=True, text=True)
    print(process.stdout)

    if process.stderr:
        print(f"[Vina System Logs]: {process.stderr}")

    with open(log_txt_file, "w") as log_file:
        log_file.write(process.stdout)

    if os.path.exists(out_pdbqt_file) and os.path.getsize(out_pdbqt_file) > 0:
        subprocess.run(["obabel", out_pdbqt_file, "-O", out_sdf_file], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        subprocess.run(["obabel", out_pdbqt_file, "-O", f"{split_pdb_prefix}.pdb", "-m"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("\n[+] ALL REQUESTED VIRTUAL SCREENING SIMULATIONS COMPLETE")

[+] Verified Rigid Receptor Target ==> 1MOX_prot_A.pdbqt
[-] Dynamic File Match: Ingesting library catalog from registry: [MONAMI.csv]
[+] Validation Injection: Native control ligand [AQ4] queued for re-docking.
[+] Vina Engine Initialized. Commencing molecular simulation tasks across 3 compounds...

 RUNNING SIMULATION FOR TARGET LIGAND COMPLEX: AQ4
AutoDock Vina v1.2.5
#################################################################
# If you used AutoDock Vina in your work, please cite:          #
#                                                               #
# J. Eberhardt, D. Santos-Martins, A. F. Tillack, and S. Forli  #
# AutoDock Vina 1.2.0: New Docking Methods, Expanded Force      #
# Field, and Python Bindings, J. Chem. Inf. Model. (2021)       #
# DOI 10.1021/acs.jcim.1c00203                                  #
#                                                               #
# O. Trott, A. J. Olson,                                        #
# AutoDock Vina: improving the s

In [54]:
#@title CELL 9: Rank Docking Scores & Validation Matrix { display-mode: "form" }
#@markdown Gather, sanitize, and rank all virtual screening binding affinities into a clean publication-ready summary spreadsheet.

import os
import subprocess
import pandas as pd

# Retrieve absolute directory pathways from environmental memory
DCK_FLD = os.environ.get('DCK_FLD')
EXP_FLD = os.environ.get('EXP_FLD')

if not DCK_FLD or not EXP_FLD:
    raise ValueError("Workspace pathways missing. Please re-run Cell 3 first to map paths.")

print("[-] Scanning docking subdirectories for completed execution log frameworks...")

# Locate all subfolders inside DOCKING directory representing evaluated hits
subfolders = [f for f in os.listdir(DCK_FLD) if os.path.isdir(os.path.join(DCK_FLD, f))]

# --- DYNAMIC REFERENCE LIGAND LOCATOR ---
crystal_ref_path = None
for folder in [DCK_FLD, EXP_FLD]:
    if os.path.exists(folder):
        candidates = [
            os.path.join(folder, f) for f in os.listdir(folder)
            if f.endswith(('.pdbqt', '.pdb')) and not "_prot_" in f and not "_output" in f
            and not os.path.isdir(os.path.join(folder, f))
        ]
        if candidates:
            crystal_ref_path = candidates[0]
            break

compiled_results = []

# --- PARSE RUNTIME LOG PACKETS ---
for ID in subfolders:
    lig_dir = os.path.join(DCK_FLD, ID)
    log_txt_path = os.path.join(lig_dir, f"{ID}_log.txt")
    out_sdf_path = os.path.join(lig_dir, f"{ID}_output.sdf")

    if not os.path.exists(log_txt_path):
        continue

    best_affinity = None
    with open(log_txt_path, 'r') as log_f:
        for line in log_f:
            parts = line.split()
            if len(parts) >= 2 and parts[0] == "1":
                try:
                    best_affinity = float(parts[1])
                    break
                except ValueError:
                    continue

    if best_affinity is None:
        continue

    rmsd_val = "N/A"

    # Calculate symmetry-corrected RMSD alignment metrics if reference matches topology
    if crystal_ref_path and os.path.exists(out_sdf_path) and os.path.getsize(out_sdf_path) > 0:
        try:
            cmd_rmsd = ["obrms", crystal_ref_path, out_sdf_path]
            rmsd_out = subprocess.run(cmd_rmsd, capture_output=True, text=True, check=True)
            output_lines = rmsd_out.stdout.splitlines()
            if output_lines:
                raw_rmsd = float(output_lines[0].split()[-1])
                rmsd_val = f"{raw_rmsd:.3f}"
        except Exception:
            rmsd_val = "Mismatch"

    compiled_results.append({
        "Ligand ID": ID,
        "Raw Affinity": best_affinity,
        "RMSD to Crystal (Å)": rmsd_val
    })

# --- RENDER AND SANITIZE SUMMARY DASHBOARD ---
if not compiled_results:
    print("\n[!] Execution Matrix Empty: No structural screening logs discovered to compile.")
    print("    ↳ Action required: Ensure CELL 8 finished running targets completely.")
else:
    # Build dataframe and sort values based on thermodynamic stability thresholds
    df_results = pd.DataFrame(compiled_results)
    df_results = df_results.sort_values(by="Raw Affinity", ascending=True).reset_index(drop=True)

    # Advanced Data Formatting Layer: Catches scientific notation explosions and formats clean decimals
    formatted_rows = []
    for idx, row in df_results.iterrows():
        raw_score = row["Raw Affinity"]

        # If affinity is an astronomical positive number, flag it as a physical atom collision
        if raw_score > 50.0:
            display_affinity = "Steric Clash"
        else:
            display_affinity = f"{raw_score:.3f}" # Force standard crisp decimal layout

        formatted_rows.append({
            "Ligand ID": row["Ligand ID"],
            "Binding Affinity (kcal/mol)": display_affinity,
            "RMSD to Crystal (Å)": row["RMSD to Crystal (Å)"]
        })

    df_clean = pd.DataFrame(formatted_rows)

    # Save clean uncompressed master spreadsheet record file directly to workspace
    master_csv_path = os.path.join(DCK_FLD, "screening_summary_report.csv")
    df_clean.to_csv(master_csv_path, index=False)

    print("\n" + "="*75)
    print("              VIRTUAL SCREENING SCORE RANKING COMPILATION MATRIX")
    print("="*75)
    print(df_clean.to_string(index=True))
    print("="*75)
    print(f"[+] Master summary dataset spreadsheet written successfully -> {master_csv_path}\n")

[-] Scanning docking subdirectories for completed execution log frameworks...

              VIRTUAL SCREENING SCORE RANKING COMPILATION MATRIX
  Ligand ID Binding Affinity (kcal/mol) RMSD to Crystal (Å)
0      SKQ1                       0.000            Mismatch
1     SKQR1                       0.000            Mismatch
2       AQ4                       0.000            Mismatch
3       NAG                Steric Clash            Mismatch
[+] Master summary dataset spreadsheet written successfully -> /content/1MOX_VS/DOCKING/screening_summary_report.csv



In [ ]:
#@title CELL 10: PLIP Interaction Profiling & Complex Generation { display-mode: "form" }
#@markdown Merge selected simulation poses with the protein matrix and extract color-coded molecular interaction tables.

Target_pose_ID = "naringenin_1" #@param {type:"string"}

import os
import pandas as pd
from plip.structure.preparation import PDBComplex

# Retrieve absolute directory pathways from environmental memory set in Cell 3
PRT_FLD = os.environ.get('PRT_FLD')
DCK_FLD = os.environ.get('DCK_FLD')
INT_FLD = os.environ.get('INT_FLD')

if not all([PRT_FLD, DCK_FLD, INT_FLD]):
    raise ValueError("Workspace pathways missing. Please re-run Cell 3 first.")

# Step 1: Parse the target layout identification string components cleanly
parts = Target_pose_ID.strip().split('_')
if len(parts) < 2 or not parts[-1].isdigit():
    raise ValueError("Invalid target format. Please enter a proper structured ID string (e.g., 'PY1_1' or 'NARINGENIN_1').")

ligand_id = "_".join(parts[:-1]).upper()
pose_num = parts[-1]

# Locate source structural input files dynamically
prot_candidates = [f for f in os.listdir(PRT_FLD) if f.endswith('.pdb') and "_prot_" in f]
if not prot_candidates:
    raise FileNotFoundError("Clean receptor backbone PDB structure missing from PROTEIN folder. Run Cell 4 first.")

receptor_pdb_path = os.path.join(PRT_FLD, prot_candidates[0])
ligand_pose_path = os.path.join(DCK_FLD, ligand_id, f"{ligand_id}_{pose_num}.pdb")

if not os.path.exists(ligand_pose_path):
    raise FileNotFoundError(f"Target conformation file '{os.path.basename(ligand_pose_path)}' not discovered. Ensure Cell 8 ran successfully.")

complex_pdb_path = os.path.join(INT_FLD, f"{Target_pose_ID.upper()}_complex.pdb")

print(f"[-] Synthesizing consolidated complex model for target frame: {Target_pose_ID.upper()}")

# Concatenate the protein backbone and ligand pose coordinates into a single unified PDB matrix file
with open(complex_pdb_path, 'w') as complex_file:
    with open(receptor_pdb_path, 'r') as r_file:
        for line in r_file:
            if not any(line.startswith(x) for x in ["END", "CONECT"]):
                complex_file.write(line)
    with open(ligand_pose_path, 'r') as l_file:
        for line in l_file:
            if not any(line.startswith(x) for x in ["END", "MASTER"]):
                complex_file.write(line)
    complex_file.write("END\n")

print(f"    ↳ Complex file generated ==> INTERACTION folder as: {os.path.basename(complex_pdb_path)}")
print(f"[-] Loading chemical complex space into the programmatic PLIP analyzer engine...")

# Step 2: Initialize the PLIP complex object structure and trigger analysis
mol_complex = PDBComplex()
mol_complex.load_pdb(complex_pdb_path)
mol_complex.analyze()

interaction_matrix = []

# Robust distance resolver helper
def safe_extract_distance(interaction_obj):
    for attr in ['distance', 'dist', 'distance_ad', 'dist_d_a', 'distance_ah']:
        if hasattr(interaction_obj, attr):
            val = getattr(interaction_obj, attr)
            if isinstance(val, (int, float)):
                return round(float(val), 2)
    return 0.0

# Dynamic fallback scanner to locate and classify lists of interactions inside the object
def adaptive_harvest_list(obj, primary_attr, search_keywords):
    # Try the standard primary property attribute first
    if hasattr(obj, primary_attr):
        val = getattr(obj, primary_attr)
        if isinstance(val, (list, tuple)):
            return val

    # Fuzzy keyword matching across all attributes if properties are renamed
    for attr in dir(obj):
        if any(key in attr.lower() for key in search_keywords):
            try:
                val = getattr(obj, attr)
                if isinstance(val, (list, tuple)):
                    return val
            except:
                pass
    return []

# Map out non-covalent structural interactions across parsed binding pockets
for site in mol_complex.interaction_sets:
    interaction_set = mol_complex.interaction_sets[site]

    # 1. Capture Hydrophobic Interactions (GREEN)
    hydrophobic_source = adaptive_harvest_list(interaction_set, 'hydrophobic_contacts', ['hydrophobic', 'hydrophob'])
    for c in hydrophobic_source:
        interaction_matrix.append({
            "RESNR": int(getattr(c, 'resnr', 0)),
            "RESTYPE": str(getattr(c, 'restype', 'UNK')),
            "DIST_CALC": safe_extract_distance(c),
            "BOND": "HYDROPHOBIC",
            "COLOR": "GREEN"
        })

    # 2. Capture Hydrogen Bonds (BLUE)
    hbond_source = adaptive_harvest_list(interaction_set, 'hbonds', ['hbond', 'hydrogen'])
    for h in hbond_source:
        interaction_matrix.append({
            "RESNR": int(getattr(h, 'resnr', 0)),
            "RESTYPE": str(getattr(h, 'restype', 'UNK')),
            "DIST_CALC": safe_extract_distance(h),
            "BOND": "HYDROGENBOND",
            "COLOR": "BLUE"
        })

    # 3. Capture Salt Bridges (YELLOW)
    salt_source = adaptive_harvest_list(interaction_set, 'saltbridges', ['salt', 'bridge'])
    for s in salt_source:
        interaction_matrix.append({
            "RESNR": int(getattr(s, 'resnr', 0)),
            "RESTYPE": str(getattr(s, 'restype', 'UNK')),
            "DIST_CALC": safe_extract_distance(s),
            "BOND": "SALTBRIDGE",
            "COLOR": "YELLOW"
        })

    # 4. Capture Pi-Stacking Interactions (PURPLE)
    pi_source = adaptive_harvest_list(interaction_set, 'pistacking', ['pi', 'stack', 'pistack'])
    for p in pi_source:
        interaction_matrix.append({
            "RESNR": int(getattr(p, 'resnr', 0)),
            "RESTYPE": str(getattr(p, 'restype', 'UNK')),
            "DIST_CALC": safe_extract_distance(p),
            "BOND": "PI-STACKING",
            "COLOR": "PURPLE"
        })

# Step 3: Construct unified Pandas DataFrame matching your exact formatting layout specifications
plip_report = pd.DataFrame(interaction_matrix)

# Clear any placeholder row artifacts if generated by fallback defaults
if not plip_report.empty:
    plip_report = plip_report[plip_report["RESNR"] != 0].sort_values(by="RESNR", ascending=True).reset_index(drop=True)

if not plip_report.empty:
    report_csv_path = os.path.join(INT_FLD, f"{Target_pose_ID.upper()}_interactions.csv")
    plip_report.to_csv(report_csv_path, index=False)

    print(f"[+] PLIP MOLECULAR PROFILING COMPLETE FOR POSE: {Target_pose_ID.upper()}")
    print(f"    ↳ Saved contact matrix data -> {report_csv_path}\n")

    display(plip_report)
else:
    print(f"\n[+] Complex parsed successfully, but no classic non-covalent contacts crossed the threshold for pose '{Target_pose_ID.upper()}'.")

In [ ]:
#@title CELL 11: Advanced 3D Binding Mode Graphics Panel { display-mode: "form" }
#@markdown Configure your interactive 3D visualization studio canvas to inspect binding pocket architecture.

Target_pose_ID = "naringenin_1" #@param {type:"string"}
Protein_Style = "cartoon" #@param ["cartoon", "line", "stick", "sphere"]
Ligand_Style = "stick" #@param ["stick", "sphere"]
Show_Interactions_Pocket = True #@param {type:"boolean"}
Surface_Type = "None" #@param ["None", "VDW", "SAS", "MS"]
Surface_Opacity = 0.6 #@param {type:"slider", min:0.0, max:1.0, step:0.1}

import os
import pandas as pd
import py3Dmol

# Retrieve absolute directory pathways from environmental memory
PRT_FLD = os.environ.get('PRT_FLD')
DCK_FLD = os.environ.get('DCK_FLD')
INT_FLD = os.environ.get('INT_FLD')

if not all([PRT_FLD, DCK_FLD, INT_FLD]):
    raise ValueError("Workspace pathways missing. Please re-run Cell 3 first.")

# Parse identification strings
parts = Target_pose_ID.strip().split('_')
if len(parts) < 2 or not parts[-1].isdigit():
    raise ValueError("Invalid target format. Enter a structured ID string (e.g., 'ESCULIN_1').")

ligand_id = "_".join(parts[:-1]).upper()
pose_num = parts[-1]

# Resolve source file targets dynamically
prot_candidates = [f for f in os.listdir(PRT_FLD) if f.endswith('.pdb') and "_prot_" in f]
ligand_pose_path = os.path.join(DCK_FLD, ligand_id, f"{ligand_id}_{pose_num}.pdb")
interactions_csv = os.path.join(INT_FLD, f"{Target_pose_ID.upper()}_interactions.csv")

if not prot_candidates:
    raise FileNotFoundError("Clean receptor structure missing. Please run Cell 4 first.")
if not os.path.exists(ligand_pose_path):
    raise FileNotFoundError(f"Docking pose structure file not found for: {Target_pose_ID.upper()}")

receptor_pdb_path = os.path.join(PRT_FLD, prot_candidates[0])

print(f"[-] Initializing WebGL 3D Studio Canvas for Target complex frame: {Target_pose_ID.upper()}")

# Initialize canvas object view frame
view = py3Dmol.view(width=700, height=550)

# Layer 0: Load and style the clean macromolecular protein backbone matrix
with open(receptor_pdb_path, 'r') as f:
    view.addModel(f.read(), 'pdb')
view.setStyle({'model': 0}, {Protein_Style: {'color': 'spectrum'}})

# Layer 1: Load and style the target chemical screening ligand pose conformation
with open(ligand_pose_path, 'r') as f:
    view.addModel(f.read(), 'pdb')

if Ligand_Style == "stick":
    view.setStyle({'model': 1}, {'stick': {'colorscheme': 'magentaCarbon', 'radius': 0.15}})
elif Ligand_Style == "sphere":
    view.setStyle({'model': 1}, {'sphere': {'colorscheme': 'magentaCarbon', 'scale': 0.3}})

# Highlight Coordinating Pocket Residues using the Data Framework from Cell 10
interacting_residues = []
if Show_Interactions_Pocket and os.path.exists(interactions_csv):
    try:
        df_bonds = pd.read_csv(interactions_csv)
        interacting_residues = df_bonds["RESNR"].dropna().unique().astype(int).tolist()
        print(f"[+] Interaction data found. Isolating and highlighting {len(interacting_residues)} coordinating binding residues...")
    except Exception as e:
        print(f"[!] Matrix parsing skipped: {e}")

if interacting_residues:
    # Explicitly render pocket contacts as thick sticks with yellow labels to match high-end graphic suites
    view.addStyle({'model': 0, 'resi': interacting_residues}, {'stick': {'colorscheme': 'grayCarbon', 'radius': 0.2}})
    for res_num in interacting_residues:
        view.addLabel(str(res_num), {'fontColor': 'yellow', 'backgroundColor': 'black', 'backgroundOpacity': 0.6}, {'model': 0, 'resi': res_num})

# Calculate and overlay electronic/molecular pocket boundary molecular surfaces
if Surface_Type != "None":
    surf_map = {"VDW": py3Dmol.VDW, "SAS": py3Dmol.SAS, "MS": py3Dmol.MS}
    # Frame surface exclusively around the ligand domain space to keep focus sharp inside the pocket
    view.addSurface(surf_map[Surface_Type], {'opacity': Surface_Opacity, 'color': 'white'}, {'model': 0})

# Center view directly onto the coordinates of the ligand hit
view.zoomTo({'model': 1})
print(f"[+] 3D CANVAS STABILIZED. Use your mouse to rotate (Left Click), pan (Right Click), or zoom (Scroll).")
view.show()

In [ ]:
#@title CELL 11.5: 2D Protein-Ligand Interaction Network Mapping { display-mode: "form" }
#@markdown Generate an elegant, publication-ready schematic network map matching commercial molecular modeling software aesthetics.

Target_pose_ID = "ESCULIN_1" #@param {type:"string"}
Canvas_Style = "White Studio" #@param ["White Studio", "Dark Mode"]
Show_Distance_Labels = True #@param {type:"boolean"}

import os
import math
import pandas as pd
import matplotlib.pyplot as plt

# Retrieve absolute directory pathways from environmental memory
INT_FLD = os.environ.get('INT_FLD')
if not INT_FLD:
    raise ValueError("Workspace pathways missing. Please re-run Cell 3 first.")

interactions_csv = os.path.join(INT_FLD, f"{Target_pose_ID.upper()}_interactions.csv")

print(f"[-] Locating structural interaction records for: {Target_pose_ID.upper()}")

if not os.path.exists(interactions_csv):
    print(f"[!] Error: No interaction CSV file found for '{Target_pose_ID.upper()}'.")
    print("    ↳ Action required: Make sure you run CELL 10 first to extract the bonds.")
else:
    df = pd.read_csv(interactions_csv)
    if df.empty:
        print(f"[!] Map Alert: The interaction matrix for '{Target_pose_ID.upper()}' is completely empty.")
    else:
        print(f"[+] Rebuilding 2D layout engine for high-resolution publication export...")
        df["RES_LABEL"] = df["RESTYPE"].str.strip() + " " + df["RESNR"].astype(str)

        # Software-matching color palette dictionary definitions
        color_palette = {
            "HYDROGENBOND": {"line": "#0284C7", "box_fill": "#E0F2FE", "box_edge": "#0369A1", "text": "#0369A1"},
            "HYDROPHOBIC":  {"line": "#16A34A", "box_fill": "#DCFCE7", "box_edge": "#15803D", "text": "#15803D"},
            "SALTBRIDGE":   {"line": "#EA580C", "box_fill": "#FFEDD5", "box_edge": "#C2410C", "text": "#C2410C"},
            "PI-STACKING":  {"line": "#9333EA", "box_fill": "#F3E8FF", "box_edge": "#7E22CE", "text": "#7E22CE"}
        }

        if Canvas_Style == "Dark Mode":
            bg_color, text_color, card_bg = "#121212", "#FFFFFF", "#1E1E1E"
            ligand_fill, ligand_text = "#4A044E", "#FDF4FF"
        else:
            bg_color, text_color, card_bg = "#FFFFFF", "#1E293B", "#F8FAFC"
            ligand_fill, ligand_text = "#701A75", "#FFFFFF"

        fig, ax = plt.subplots(figsize=(9, 9), facecolor=bg_color)
        ax.set_facecolor(bg_color)

        ligand_x, ligand_y = 0.0, 0.0
        unique_residues = df["RES_LABEL"].unique().tolist()
        num_residues = len(unique_residues)

        residue_coords = {}
        for idx, res in enumerate(unique_residues):
            # Alternating radial layout to eliminate crowding completely
            current_radius = 4.8 if idx % 2 == 0 else 3.8
            angle = (2 * math.pi * idx) / num_residues
            rx = current_radius * math.cos(angle)
            ry = current_radius * math.sin(angle)
            residue_coords[res] = (rx, ry)

        # Step 1: Draw high-contrast non-covalent vector lines
        bond_counters = {}
        for _, row in df.iterrows():
            res = row["RES_LABEL"]
            rx, ry = residue_coords[res]
            bond_type = row["BOND"]
            distance = row["DIST_CALC"]
            b_cfg = color_palette.get(bond_type, {"line": "#64748B"})

            if bond_type == "HYDROGENBOND":
                linestyle = (0, (4, 3))
                linewidth = 2.5
            elif bond_type == "HYDROPHOBIC":
                linestyle = "solid"
                linewidth = 2.0
            elif bond_type == "SALTBRIDGE":
                linestyle = (0, (6, 2, 1, 2))
                linewidth = 3.0
            else:
                linestyle = "dotted"
                linewidth = 2.5

            ax.plot([ligand_x, rx], [ligand_y, ry], color=b_cfg["line"], linestyle=linestyle, linewidth=linewidth, zorder=1)

            if Show_Distance_Labels:
                mx = (ligand_x + rx) / 2.0
                my = (ligand_y + ry) / 2.0
                bond_counters[res] = bond_counters.get(res, 0) + 1
                if bond_counters[res] > 1:
                    mx += 0.22 * (bond_counters[res] - 1)
                    my += 0.22 * (bond_counters[res] - 1)
                ax.text(mx, my, f"{distance} Å", color=b_cfg["line"], fontsize=9, fontweight="bold",
                        bbox=dict(boxstyle="square,pad=0.2", facecolor=bg_color, edgecolor="none", alpha=0.9),
                        ha="center", va="center", zorder=3)

        # Step 2: Draw dynamic rounded residue cards
        for res, (rx, ry) in residue_coords.items():
            res_df = df[df["RES_LABEL"] == res]
            primary_bond = res_df["BOND"].iloc[0]
            b_cfg = color_palette.get(primary_bond, {"box_fill": "#F1F5F9", "box_edge": "#94A3B8", "text": "#475569"})
            ax.text(rx, ry, res, color=b_cfg["text"], fontsize=10, fontweight="bold",
                    ha="center", va="center", zorder=4,
                    bbox=dict(boxstyle="round,pad=0.4", facecolor=b_cfg["box_fill"], edgecolor=b_cfg["box_edge"], linewidth=1.5))

        # Step 3: Draw central ligand structural hub panel
        lig_hub_label = Target_pose_ID.split('_')[0].upper()
        ax.text(ligand_x, ligand_y, lig_hub_label, color=ligand_text, fontsize=12, fontweight="bold",
                ha="center", va="center", zorder=5,
                bbox=dict(boxstyle="round,pad=0.7", facecolor=ligand_fill, edgecolor=text_color, linewidth=2.0))

        ax.set_xlim(-6.2, 6.2)
        ax.set_ylim(-6.2, 6.2)
        ax.axis("off")

        # Compact and safe Legend Drawing Sub-engine
        legend_elements = []
        active_bonds = list(df["BOND"].unique())
        for b_name in active_bonds:
            if b_name in color_palette:
                b_cfg = color_palette[b_name]
                b_style = (0, (4, 3)) if b_name == "HYDROGENBOND" else "solid" if b_name == "HYDROPHOBIC" else (0, (6, 2, 1, 2)) if b_name == "SALTBRIDGE" else "dotted"
                legend_elements.append(plt.Line2D([0], [0], color=b_cfg["line"], linestyle=b_style, lw=2.5, label=b_name))

        ax.legend(handles=legend_elements, loc="upper right", frameon=True, facecolor=bg_color, edgecolor=text_color, labelcolor=text_color, fontsize=10)

        plt.title(f"2D PROTEIN-LIGAND INTERACTION PROFILE\nTarget Complex: {Target_pose_ID.upper()}", color=text_color, fontsize=13, fontweight="bold", pad=15)

        output_png_path = os.path.join(INT_FLD, f"{Target_pose_ID.upper()}_2D_interaction.png")
        plt.savefig(output_png_path, dpi=300, bbox_inches="tight", facecolor=fig.get_facecolor(), edgecolor="none")

        print(f"[+] Production-grade schematic compiled successfully.")
        print(f"    ↳ Saved 2D graphic -> {output_png_path}\n")
        plt.show()

In [ ]:
#@title CELL 12: Permanent Cloud Archiving & Export { display-mode: "form" }
#@markdown Securely package and back up your complete virtual screening and interaction profile workspace.

Export_Target_Channel = "Local Download via Browser" #@param ["Export to Mounted Google Drive", "Local Download via Browser"]

import os
import shutil

# Retrieve absolute directory pathways from environmental memory set in Cell 3
WRK_DIR = os.environ.get('WRK_DIR')
if not WRK_DIR:
    raise ValueError("Active project workspace root not discovered. Please run Cell 3 first to map paths.")

job_id_string = os.path.basename(WRK_DIR)
archive_output_name = f"{job_id_string}_final_export"
archive_local_path = os.path.join(os.getcwd(), archive_output_name)

print(f"[-] Compressing workspace tree node elements for project: [{job_id_string}]")

# Compress the entire project root directory folder recursively into a unified .zip archive
shutil.make_archive(archive_local_path, 'zip', WRK_DIR)
zip_file_complete = f"{archive_local_path}.zip"

print(f"    ↳ Archive compilation complete: {os.path.basename(zip_file_complete)}")

# --- ROUTE A: Secure Mount & Transfer to Personal Google Cloud Storage ---
if Export_Target_Channel == "Export to Mounted Google Drive":
    print("[-] Activating Google Drive authentication pathways...")
    try:
        from google.colab import drive
        # Triggers secure handshake loop inside the Colab platform workspace
        drive.mount('/content/drive', force_remount=True)

        gdrive_destination_folder = "/content/drive/MyDrive/Nanomed_Docking_Results"
        os.makedirs(gdrive_destination_folder, exist_ok=True)

        gdrive_target_path = os.path.join(gdrive_destination_folder, f"{archive_output_name}.zip")
        shutil.copy(zip_file_complete, gdrive_target_path)

        print("\n[+] CLOUD PERSISTENCE ARCHIVING COMPLETE")
        print(f"    ↳ Target File Synchronized -> {gdrive_target_path}")
        print("    ↳ Status: Securely saved inside your personal Google Drive storage node folder.")
    except Exception as e:
        print(f"\n[!] Cloud Link Failure: {e}")
        print("    ↳ Fallback action: Defaulting back to native browser storage pipeline...")
        Export_Target_Channel = "Local Download via Browser"

# --- ROUTE B: Native Direct Browser File Stream Extraction ---
if Export_Target_Channel == "Local Download via Browser":
    print("[-] Initializing secure browser download streaming channels...")
    try:
        from google.colab import files
        print("\n[+] CLICK PROMPT TRIGGER DOWNSTREAM REACHED")
        print("    ↳ Your browser will now prompt you to save the compiled ZIP dataset archive.")
        files.download(zip_file_complete)
    except Exception as e:
        print(f"\n[!] Browser Stream Error: {e}")
        print(f"    ↳ Action required: Manually right-click the file in the file explorer and download: {zip_file_complete}")